## RAG Pipelines- Data Ingestion to Vector DB Pipeline

In [22]:
import os
from pathlib import Path
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader #PDF ko read/load karne ke liye LangChain ka loader hai.
from langchain_text_splitters import RecursiveCharacterTextSplitter #chunking ke liye


#### STEP 1 : DATA Ingestion (pdf loading + pdf -> document structure) 

In [23]:
### Read all the pdf's inside the directory

def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)
    
    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    
    print(f"Found {len(pdf_files)} PDF files to process")
    
    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
            
            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
            
            all_documents.extend(documents)
            print(f"  ✓ Loaded {len(documents)} pages")
            
        except Exception as e:
            print(f"  ✗ Error: {e}")
    
    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

# Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("../data")

Found 2 PDF files to process

Processing: attention.pdf
  ✓ Loaded 11 pages

Processing: company.pdf
  ✓ Loaded 2 pages

Total documents loaded: 13


In [24]:
all_pdf_documents

[Document(metadata={'producer': 'PyPDF2', 'creator': 'PyPDF', 'creationdate': '', 'subject': 'Neural Information Processing Systems http://nips.cc/', 'publisher': 'Curran Associates, Inc.', 'language': 'en-US', 'created': '2017', 'eventtype': 'Poster', 'description-abstract': 'The dominant sequence transduction models are based on complex recurrent orconvolutional neural networks in an encoder and decoder configuration. The best performing such models also connect the encoder and decoder through an attentionm echanisms.  We propose a novel, simple network architecture based solely onan attention mechanism, dispensing with recurrence and convolutions entirely.Experiments on two machine translation tasks show these models to be superiorin quality while being more parallelizable and requiring significantly less timeto train. Our single model with 165 million parameters, achieves 27.5 BLEU onEnglish-to-German translation, improving over the existing best ensemble result by over 1 BLEU. On 

#### STEP 2 : Chunking

In [25]:
### Text splitting get into chunks

def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,#Adjacent chunks me approximately 200 characters overlap rakhne ki setting
        length_function=len,
        separators=["\n\n", "\n", " ", ""]#Recursive splitter ko bataya ja raha hai ki text ko kahan break karne ki koshish karni hai
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    # Show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs

In [26]:
chunks = split_documents(all_pdf_documents)
chunks


Split 13 documents into 47 chunks

Example chunk:
Content: Attention Is All You Need
Ashish Vaswani∗
Google Brain
avaswani@google.com
Noam Shazeer∗
Google Brain
noam@google.com
Niki Parmar∗
Google Research
nikip@google.com
Jakob Uszkoreit∗
Google Research
usz...
Metadata: {'producer': 'PyPDF2', 'creator': 'PyPDF', 'creationdate': '', 'subject': 'Neural Information Processing Systems http://nips.cc/', 'publisher': 'Curran Associates, Inc.', 'language': 'en-US', 'created': '2017', 'eventtype': 'Poster', 'description-abstract': 'The dominant sequence transduction models are based on complex recurrent orconvolutional neural networks in an encoder and decoder configuration. The best performing such models also connect the encoder and decoder through an attentionm echanisms.  We propose a novel, simple network architecture based solely onan attention mechanism, dispensing with recurrence and convolutions entirely.Experiments on two machine translation tasks show these models to be superiorin

[Document(metadata={'producer': 'PyPDF2', 'creator': 'PyPDF', 'creationdate': '', 'subject': 'Neural Information Processing Systems http://nips.cc/', 'publisher': 'Curran Associates, Inc.', 'language': 'en-US', 'created': '2017', 'eventtype': 'Poster', 'description-abstract': 'The dominant sequence transduction models are based on complex recurrent orconvolutional neural networks in an encoder and decoder configuration. The best performing such models also connect the encoder and decoder through an attentionm echanisms.  We propose a novel, simple network architecture based solely onan attention mechanism, dispensing with recurrence and convolutions entirely.Experiments on two machine translation tasks show these models to be superiorin quality while being more parallelizable and requiring significantly less timeto train. Our single model with 165 million parameters, achieves 27.5 BLEU onEnglish-to-German translation, improving over the existing best ensemble result by over 1 BLEU. On 

### Embedding(Chunks to Embedding)

In [27]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

- def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
__init__() constructor hai.
Jab tum:
embedding_manager = EmbeddingManager()
likhoge, ye automatically execute hoga.


- 

In [28]:
class EmbeddingManager: #Embedding model ko load karna aur text ke embeddings generate karna.
    """Handles document embedding generation using SentenceTransformer"""
    
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"): #Sentence Transformer embedding model
        """
        Initialize the embedding manager
        
        Args:
            model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None#Abhi model load nahi hua hai.
        self._load_model()#Constructor ke andar hi _load_model() call kar diya.

    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)# model load
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")#Har text ka embedding vector kitne numbers ka hoga?isme 384 dimensions h
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise #Sirf error print karke silently continue nahi karta. Original error ko dobara raise karta hai.

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:#List of texts → NumPy array of embeddings, input type list of string  [[0.1, 0.2, ...],[0.3, 0.4, ...],[0.5, 0.6, ...]]
        """
        Generate embeddings for a list of texts
        
        Args:
            texts: List of text strings to embed
            
        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model: #Check kar raha hai ki model loaded hai ya nahi.
            raise ValueError("Model not loaded")
        print(f"Generating embeddings for {len(texts)} texts...")#texts = [ "text1", "text2", "text3"] ---> Output:Generating embeddings for 3 texts...
        embeddings = self.model.encode(texts, show_progress_bar=True)#actual embedding generation and shows progress bar while genearting
        print(f"Generated embeddings with shape: {embeddings.shape}")# returns:(500, 384) ===>500 → number of texts/chunks, 384 → embedding dimension
        return embeddings #Embedding matrix return kar raha hai.


## initialize the embedding manager

embedding_manager=EmbeddingManager()
embedding_manager

Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8004.40it/s]


Model loaded successfully. Embedding dimension: 384


C:\Users\Admin\AppData\Local\Temp\ipykernel_18204\492378892.py:20: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")#Har text ka embedding vector kitne numbers ka hoga?isme 384 dimensions h


### VectorStore(store the embed in vectordb)

In [29]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""
    
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """
        Initialize the vector store
        
        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name #ChromaDB ke andar ek collection hogi.
        self.persist_directory = persist_directory#ChromaDB ka persistent data disk par kahan store hoga
        self.client = None #initially none
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)#Directory create karna
            #Data memory me temporarily rakhne ke bajay disk par persist hoga
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            
            # Get or create collection
            #Collection exist karti hai? -> YES → get it -> NO → create it
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store
        
        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        #Length check=>Because har document ke liye exactly ek embedding chahiye.
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        print(f"Adding {len(documents)} documents to vector store...")
        
        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
        
        #documents aur embeddings ko ek-ek karke pair karo, aur har pair par loop chalao.
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID for chunk
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            
            # Prepare metadata
            metadata = dict(doc.metadata)#Document ki existing metadata ki copy banayi.
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            
            # Document content
            documents_text.append(doc.page_content)#Actual chunk text ChromaDB ke liye collect kar rahe ho.
            
            # Embedding
            embeddings_list.append(embedding.tolist())#embedding numpy array ko python list me convert to add for chromadb

          # Add to collection(chromadb me add)
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vectorstore=VectorStore()
vectorstore   


Vector store initialized. Collection: pdf_documents
Existing documents in collection: 94


- documents aur embeddings ko ek-ek karke pair karo, aur har pair par loop chalao.
- Loop
 for i, (doc, embedding) in enumerate(
    zip(documents, embeddings)
):

- zip()
 Suppose:documents: D1 D2 D3
embeddings: E1 E2 E3

- zip() pairs banata hai:
(D1, E1)
(D2, E2)
(D3, E3)
So correct document ka correct embedding milta hai.
- enumerate()
 Index bhi deta hai:
0 → (D1, E1)
1 → (D2, E2)
2 → (D3, E3)
So: i index hai.

In [30]:
### Chunks ka sirf text nikalo
texts=[doc.page_content for doc in chunks]

## Text ko embeddings mein convert karo

embeddings=embedding_manager.generate_embeddings(texts)

##store int he vector db(chromaDB)
vectorstore.add_documents(chunks,embeddings)

Generating embeddings for 47 texts...


Batches: 100%|██████████| 2/2 [00:02<00:00,  1.01s/it]

Generated embeddings with shape: (47, 384)
Adding 47 documents to vector store...
Successfully added 47 documents to vector store
Total documents in collection: 141


### Retriever Pipeline From VectorStore
##### query(embeded) + vectordb = context to llm 
##### retrive the context from query and KB

In [37]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""
    
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever
        
        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query
        
        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold
            
        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")
        
        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]

        # Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )
            
            # Process results
            retrieved_docs = []
            
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]
                
                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance
                    
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })
                
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")
            
            return retrieved_docs

        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

rag_retriever=RAGRetriever(vectorstore,embedding_manager)
rag_retriever    

In [49]:
results = rag_retriever.retrieve(
    "What is the leave policy?",
    top_k=5
)
results

Retrieving documents for query: 'What is the leave policy?'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 87.11it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)


[{'id': 'doc_f6101944_43',
  'content': 'Company Employee Handbook\nSample Document for RAG Data Ingestion Pipeline\n1. Leave Policy\nEmployees receive 18 paid leaves per calendar year. Casual leave can be used for personal\nrequirements, while sick leave may be used when an employee is unwell. Employees should apply for\nplanned leave at least three working days in advance through the HR portal.\n2. Work From Home Policy\nEmployees may work from home up to two days per week with manager approval. Remote employees\nmust remain available during core working hours from 10:00 AM to 6:00 PM. Employees should maintain a\nstable internet connection and attend scheduled meetings.\n3. Attendance\nThe standard working week is Monday to Friday. Employees are expected to complete eight working hours\nper day. Late arrival of more than 30 minutes should be communicated to the reporting manager.\nRepeated unexplained absence may result in disciplinary action.\n4. Employee Benefits',
  'metadata': {

In [56]:

res = rag_retriever.retrieve(
    "What is attention is all you need?",
    top_k=5
)

Retrieving documents for query: 'What is attention is all you need?'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 87.97it/s]

Generated embeddings with shape: (1, 384)
Retrieved 0 documents (after filtering)
